In [50]:
import fitz
import pandas as pd
import numpy as np
import json

from sklearn.metrics.pairwise import cosine_similarity

from ollama import chat
from ollama import embed

In [51]:
import os
os.makedirs("../outputs", exist_ok=True)

In [2]:
pdf_path = r"C:\Users\Hp\Desktop\Data Analysis with AI\Assignment 1\uzbekistan2006en.pdf"

doc = fitz.open(pdf_path)

text = ""

for page in doc:
    text += page.get_text()

print("Text length : ", len(text))
print(text)

Text length :  429192
2
UNDP is the UN’s global development network, advocating for change and connecting countries to knowledge, experience 
and resources to help people build a better life. We are on the ground in 166 countries, working with them on their own solu-
tions to global and national development challenges. As they develop local capacity, they draw on the people of UNDP and 
our wide range of partners.
© Report materials can not be reproduced in whole or in part, without prior permission of UNDP.
The analysis and recommendations expressed in this report do not necessarily represent the views of UNDP. This Report is an 
independent publication produced by groups of experts commissioned by UNDP.
© United Nations Development Program  
Сountry Office for Uzbekistan
700029 Tashkent , Shevchenko str,  4
Tel: + 998 71 120 34 50
Fax: + 998 71 120 34 85
www.undp.uz 

National Human Development Report 2006: Health For All

NATIONAL HUMAN DEVELOPMENT REPORT TEAM
TEAM LEADERS  
Galin

In [3]:
def create_chunks(text,
                  chunk_size=1500,
                  overlap=200):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunks.append(text[start:end])

        start += chunk_size - overlap

    return chunks

In [4]:
chunks = create_chunks(text)

print("Chunks : ", len(chunks))

Chunks :  331


In [5]:
from ollama import embed

response = embed(
    model="nomic-embed-text",
    input=chunks[0]
)

print(len(response["embeddings"][0]))

768


In [6]:
embeddings = []

for chunk in chunks:

    e = embed(
        model="nomic-embed-text",
        input=chunk
    )

    embeddings.append(e["embeddings"][0])

In [ ]:
embeddings = np.array(embeddings)

In [48]:
retrieve(
    "health reforms in Uzbekistan"
)

['public of Uzbekistan of 25.01.2005  No 30 “On  State \nProgramme “The Year of Health”\nOrdinance of the Cabinet of Ministers of the Republic of Uzbekistan of 21.12.2005  No 276 “On approv-\ning the improved system of remuneration of labour of health providers” \nOrdinance of the Cabinet of Ministers of the Republic of Uzbekistan of 28.09.2005  No 217 “On measures \nto further reform  the system of financing and managing health care institutions in the Republic of \nUzbekistan” \nOrdinance of the Cabinet of Ministers of the Republic of Uzbekistan of 08.06.2004  No 264 “On measures \nto complete the experiment and extend reforms in the health care system” \nOrdinance  of the Cabinet of Ministers of the Republic of Uzbekistan of 23.04.2004  No 195 “On mea-\nsures to further develop the   Public System “Mother and Child Screening” \nOrdinance of the Cabinet of Ministers of the Republic of Uzbekistan of 25.08.2003  No 365 “  On \napproval of the Statute on medical examination of people wh

In [49]:
context = "\n".join(
    retrieve(
        "health reforms in Uzbekistan"
    )
)

In [12]:
prompt = f"""
Using ONLY the context below.

Summarise the health reforms.

Context:

{context}
"""

In [13]:
response = chat(
    model="llama3.2",
    messages=[
        {
            "role":"user",
            "content":prompt
        }
    ]
)

print(response["message"]["content"])

Unfortunately, there is no mention of "health reforms" in the provided context. However, based on the information about the health care system and laws related to it, I can summarize some key aspects of the system:

* The health care system in Uzbekistan combines features of the Soviet health care system with modern elements.
* One of the primary goals is to ensure unlimited access to primary health care services for all population groups, including rural areas.
* Rural health care services were previously provided by medical attendants (feldshers), but under the new system, qualified health care staff with medical university degrees will provide assistance in these areas.
* There are laws and regulations related to various aspects of health care, such as mental health care, prevention of disease caused by HIV, and protection against tuberculosis.

If you're looking for information on specific "health reforms," I would need more context or information about what you mean by "reforms."


In [14]:
from ollama import chat

def ask_llm(prompt, model="llama3.2"):
    
    response = chat(
        model=model,
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    return response["message"]["content"]

In [15]:
chapter_query = "Chapter 1 human development"

retrieved_chunks = retrieve(
    chapter_query,
    top_k=5
)

retrieved_context = "\n".join(
    retrieved_chunks
)

In [16]:
summary_prompt = f"""
You are a UN development analyst.

Using ONLY the provided context.

Provide:

1. Executive summary (150 words maximum)

2. Five key findings

3. Five key policy priorities

Context:

{context}
"""

In [17]:
report_summary = ask_llm(summary_prompt)

In [18]:
chapter_summaries = {}

chapter_queries = {
    "Chapter 1": "human development in Uzbekistan",
    "Chapter 2": "health situation in Uzbekistan",
    "Chapter 3": "healthcare reforms in Uzbekistan",
    "Chapter 4": "millennium development goals Uzbekistan"
}

for chapter, query in chapter_queries.items():

    retrieved_context = "\n".join(
        retrieve(query, top_k=5)
    )

    prompt = f"""
    You are a UN development analyst.

    Summarise this chapter in less than 100 words.

    Focus on:
    - key findings
    - health outcomes
    - development challenges
    - important policies

    Context:
    {retrieved_context}
    """

    summary = ask_llm(prompt)

    chapter_summaries[chapter] = summary

In [19]:
for chapter, summary in chapter_summaries.items():
    print("\n" + "="*50)
    print(chapter)
    print("="*50)
    print(summary)


Chapter 1
Here is a summary of the chapter in less than 100 words:

Key findings:

* Uzbekistan's economic reforms have led to a 7% growth rate in GDP and increased per capita income.
* The Human Development Index (HDI) has been increasing, but six former Soviet Republics, including Uzbekistan, have lower HDI scores.

Health outcomes:

* Child mortality rates have decreased significantly since the mid-1990s.
* Mother and child health programs have been implemented to address specific health concerns.

Development challenges:

* The country still faces significant development challenges, including high infant and maternal mortality rates.

Important policies:

* Economic reforms aim to promote private sector growth and investment.

Chapter 2
Here is a summary of Chapter 3: Public Health Care in Uzbekistan:

**Key Findings:**

* The Republic of Uzbekistan has a relatively long life expectancy, with an average of 72.5 years in 2004.
* Morbidity rates are high for infectious and parasitic

In [20]:
import json

with open("../outputs/chapter_summaries.json",
          "w",
          encoding="utf-8") as f:

    json.dump(
        chapter_summaries,
        f,
        indent=4,
        ensure_ascii=False
    )

In [21]:
theme_context = "\n".join(
    retrieve(
        "education health inequality economy gender climate employment",
        top_k=10
    )
)

theme_prompt = f"""
Analyse the report.

For each theme assign a score from 0 to 10.

Themes:

education
health
inequality
economy
gender
climate
employment

Return ONLY JSON.

Example:

{{
"education":8,
"health":10,
"inequality":6,
"economy":7,
"gender":5,
"climate":2,
"employment":4
}}

Context:
{theme_context}
"""

theme_scores = ask_llm(theme_prompt)

print(theme_scores)

{
"education":8,
"health":10,
"inequality":6,
"economy":7,
"gender":5,
"climate":2,
"employment":4
}


In [22]:
with open("../outputs/theme_scores.txt",
          "w",
          encoding="utf-8") as f:

    f.write(theme_scores)

In [23]:
strength_context = "\n".join(
    retrieve(
        "strengths challenges development achievements problems",
        top_k=10
    )
)

strength_prompt = f"""
Identify:

1. Development strengths (maximum 8)

2. Development challenges (maximum 8)

Return JSON only.

Context:

{strength_context}
"""

strengths_challenges = ask_llm(strength_prompt)

print(strengths_challenges)

```
{
  "development_strengths": [
    "Improvements in access to clean water",
    "Reduced child mortality rates",
    "Increased life expectancy",
    "Advances in education",
    "Growing income and economic growth",
    "Improved healthcare systems",
    "Enhanced social services",
    "Increased participation in sports and physical activities"
  ],
  "development_challenges": [
    "High levels of poverty and inequality",
    "Micronutrient deficiencies and malnutrition",
    "Environmental degradation and pollution",
    "Limited access to quality education",
    "Insufficient healthcare infrastructure",
    "Vicious cycle of poverty and poor health",
    "Lack of access to clean water and sanitation",
    "Inadequate healthcare services for vulnerable populations"
  ]
}
```


In [24]:
with open("../outputs/strengths_challenges.txt",
          "w",
          encoding="utf-8") as f:

    f.write(strengths_challenges)

In [25]:
indicator_context = "\n".join(
    retrieve(
        "HDI life expectancy literacy GDP population indicators",
        top_k=10
    )
)

indicator_prompt = f"""
Extract development indicators.

Return ONLY JSON.

{{
"HDI":"",
"Life_Expectancy":"",
"Literacy_Rate":"",
"GDP_Per_Capita":"",
"Population":"",
"GDI":"",
"Women_Empowerment_Index":""
}}

Context:

{indicator_context}
"""

indicators = ask_llm(indicator_prompt)

print(indicators)

{"HDI":0.694,"Life_Expectancy":"72.5","Literacy_Rate":"","GDP_Per_Capita":"","Population":"","GDI":"","Women_Empowerment_Index":""}


In [26]:
with open("../outputs/indicators.txt",
          "w",
          encoding="utf-8") as f:

    f.write(indicators)

In [27]:
models = [
    "llama3.2",
    "qwen2.5:3b",
    "phi3:mini"
]

model_outputs = {}

comparison_prompt = f"""
Summarise the main development findings
of the Uzbekistan 2006 report.

Context:

{context}
"""

for model in models:

    result = ask_llm(
        comparison_prompt,
        model=model
    )

    model_outputs[model] = result

    print("\n")
    print("="*50)
    print(model)
    print("="*50)
    print(result)



llama3.2
The 2006 report on Uzbekistan's health care system appears to be a comprehensive document that outlines the country's progress in improving its health care infrastructure. Based on the provided context, here are some key development findings from the report:

1. **Retaining features of Soviet and modern health care systems**: Uzbekistan's current health care system combines elements of both the Soviet and modern health care reforms.
2. **Improving access to primary health care**: The new health care system aims to provide unlimited access to primary health care, which is free-of-charge, even in a market economy.
3. **Enhancing rural health care services**: The report highlights the importance of improving the quality of rural health care services, particularly after the previous system's discrimination against rural populations.
4. **Creating new health care institutions**: New health care institutions, such as rural doctors' posts, are being established to provide medical a

In [28]:
with open("../outputs/model_outputs.json",
          "w",
          encoding="utf-8") as f:

    json.dump(
        model_outputs,
        f,
        indent=4,
        ensure_ascii=False
    )

In [29]:
llama_output = model_outputs["llama3.2"]

evaluation_prompt = f"""
You are evaluating an AI-generated summary.

Source Context:

{context[:6000]}

Generated Summary:

{llama_output}

Score:

1. Consistency (1-5)
2. Completeness (1-5)
3. Factual Alignment (1-5)

Return JSON only.
"""

evaluation = ask_llm(
    evaluation_prompt,
    model="qwen2.5:3b"
)

print(evaluation)

{
  "consistency": 4,
  "completeness": 4,
  "factual_alignment": 4
}


In [30]:
with open("../outputs/evaluation.txt",
          "w",
          encoding="utf-8") as f:

    f.write(evaluation)

In [31]:
print(type(report_summary))
print(type(chapter_summaries))
print(type(theme_scores))
print(type(indicators))

<class 'str'>
<class 'dict'>
<class 'str'>
<class 'str'>


In [32]:
import os

os.makedirs("../outputs", exist_ok=True)

In [33]:
hdi_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "HDI":[0.715,0.736,0.740,0.742,0.747,0.756]
})

life_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "LifeExpectancy":[69.1,70.8,71.3,71.2,71.6,72.5]
})

gdi_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "GDI":[0.704,0.729,0.734,0.738,0.742,0.752]
})

In [34]:
hdi_df.to_csv("../outputs/hdi_trend.csv",index=False)
life_df.to_csv("../outputs/life_expectancy.csv",index=False)
gdi_df.to_csv("../outputs/gdi_trend.csv",index=False)

In [35]:
comparison_df = pd.DataFrame({
    "Model":[
        "llama3.2",
        "qwen2.5:3b",
        "phi3:mini"
    ],
    "Output_Length":[
        len(model_outputs["llama3.2"]),
        len(model_outputs["qwen2.5:3b"]),
        len(model_outputs["phi3:mini"])
    ]
})

comparison_df

,Model,Output_Length
0,llama3.2,2380
1,qwen2.5:3b,2441
2,phi3:mini,3145


In [40]:
theme_df = pd.DataFrame({
    "Theme":[
        "Education",
        "Health",
        "Inequality",
        "Economy",
        "Gender",
        "Climate",
        "Employment"
    ],
    "Score":[8,10,6,7,5,2,4]
})

theme_df

,Theme,Score
0,Education,8
1,Health,10
2,Inequality,6
3,Economy,7
4,Gender,5
5,Climate,2
6,Employment,4


In [41]:
indicator_df = pd.DataFrame({
    "Indicator":[
        "HDI",
        "GDP Per Capita",
        "Literacy Rate",
        "Education Index",
        "Life Expectancy Index"
    ],
    "Value":[
        0.756,
        2668.1,
        99.3,
        0.91,
        0.777
    ]
})

indicator_df

,Indicator,Value
0,HDI,0.756
1,GDP Per Capita,2668.100
2,Literacy Rate,99.300
3,Education Index,0.910
4,Life Expectancy Index,0.777


In [42]:
theme_df.to_csv(
    "../outputs/theme_distribution.csv",
    index=False
)

indicator_df.to_csv(
    "../outputs/indicator_values.csv",
    index=False
)

In [43]:
evaluation_df = pd.DataFrame({
    "Metric":[
        "Consistency",
        "Completeness",
        "Factual Alignment"
    ],
    "Score":[4,4,4]
})

evaluation_df

,Metric,Score
0,Consistency,4
1,Completeness,4
2,Factual Alignment,4


In [44]:
evaluation_df.to_csv(
    "../outputs/evaluation_scores.csv",
    index=False
)

In [36]:
comparison_df.to_csv(
    "../outputs/model_comparison.csv",
    index=False
)

In [37]:
hdi_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "HDI":[0.715,0.736,0.740,0.742,0.747,0.756]
})

In [38]:
life_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "LifeExpectancy":[69.1,70.8,71.3,71.2,71.6,72.5]
})

In [45]:
gdi_df = pd.DataFrame({
    "Year":[1995,2000,2001,2002,2003,2004],
    "GDI":[0.704,0.729,0.734,0.738,0.742,0.752]
})

In [53]:
results_overview = pd.DataFrame({
    "Metric":[
        "HDI",
        "GDP Per Capita",
        "Literacy Rate",
        "Education Index",
        "Life Expectancy Index"
    ],
    "Value":[
        0.756,
        2668.1,
        99.3,
        0.91,
        0.777
    ]
})

results_overview

,Metric,Value
0,HDI,0.756
1,GDP Per Capita,2668.100
2,Literacy Rate,99.300
3,Education Index,0.910
4,Life Expectancy Index,0.777


In [54]:
results_overview.to_csv(
    "../outputs/results_overview.csv",
    index=False
)

In [46]:
print("\n===== REPORT SUMMARY =====")
print(report_summary)

print("\n===== THEME SCORES =====")
print(theme_scores)

print("\n===== INDICATORS =====")
print(indicators)

print("\n===== STRENGTHS & CHALLENGES =====")
print(strengths_challenges)

print("\n===== EVALUATION =====")
print(evaluation)


===== REPORT SUMMARY =====
Here are the requested items based on the provided context:

**Executive Summary (150 words maximum)**

The Republic of Uzbekistan's health care system is undergoing significant reforms, guided by principles of universal access, free primary health care, and improved quality of rural health services. The government has established new institutions, such as rural doctors' posts, to replace traditional medical attendants and provide better healthcare to rural populations. The 2006 "Year of Charity and Health Workers" aimed to enhance public health through integrated goal-oriented functions, focusing on preventive health care, healthy lifestyles, and improved quality of healthcare services. Despite retaining elements from the Soviet system, Uzbekistan's new system incorporates modern reforms, emphasizing human development and accessibility.

**Five Key Findings**

1. **Universal Access**: The government aims to provide universal access to primary health care, r